# Natural Language Inference

This notebook trains a non-transformer NLI classifier for **binary entailment** using:

- Pretrained **FastText** word embeddings (300d)
- An **ESIM-lite** style architecture:
  - shared BiLSTM encoder for premise & hypothesis
  - soft alignment (attention)
  - local inference features (concat, difference, product)
  - pooling + MLP classifier

In [7]:
import os

# Path to the training and validation datasets (used for training)
BASE_DIR = ".."

DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
DEV_CSV   = os.path.join(DATA_DIR, "dev.csv")

# Output directory containing the saved training artifacts
OUT_DIR  = os.path.join(BASE_DIR, "outputs/category_B")
os.makedirs(OUT_DIR, exist_ok=True)

print("TRAIN_CSV:", TRAIN_CSV)
print("DEV_CSV:", DEV_CSV)
print("OUT_DIR:", OUT_DIR)

TRAIN_CSV: ./data/train.csv
DEV_CSV: ./data/dev.csv
OUT_DIR: ./outputs/category_B


## Install dependencies

We use TensorFlow (Keras), NumPy/Pandas, scikit-learn metrics, and gensim for downloading FastText embeddings.

In [8]:
!pip -q install tensorflow pandas numpy scikit-learn gensim tqdm

## Imports

Required modules/libraries used in the training

In [9]:
# imports
import json
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
import random

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Dropout,
    Concatenate, Multiply, Subtract,
    GlobalMaxPooling1D, GlobalAveragePooling1D,
    LayerNormalization, Attention
)
from tensorflow.keras.models import Model

import gensim.downloader as api


## Configuration

We fix seeds for reproducibility and define main hyperparameters:

- `VOCAB_SIZE`: maximum vocabulary size for tokenizer
- `EMBED_DIM`: embedding dimension (FastText 300d)
- `BATCH_SIZE`, `EPOCHS`: training parameters

In [10]:
SEED = 42 #reproducibility
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

VOCAB_SIZE = 30000 #Vocabulary size has been set to 30,000 as it covers most of the frequent words.
EMBED_DIM  = 300
BATCH_SIZE = 32
EPOCHS     = 15

##  Load train data

Read the csv into a dataframe.

Split the dataframe into required columns.

In [11]:
def load_data(csv_path):
    """
    Load NLI dataset from CSV file.
    Returns dataframe, premise column, hypothesis column, and labels.
    """
    df = pd.read_csv(csv_path)

    premises = df["premise"].astype(str)
    hypotheses = df["hypothesis"].astype(str)
    labels = df["label"].astype(int).values

    return df, premises, hypotheses, labels

In [12]:
train_df, train_premises, train_hypotheses, y_train = load_data(TRAIN_CSV)
dev_df, dev_premises, dev_hypotheses, y_dev = load_data(DEV_CSV)

print("train:", train_df.shape, "dev:", dev_df.shape)
print("train label dist:", train_df["label"].value_counts(normalize=True).to_dict())

train: (24432, 3) dev: (6736, 3)
train label dist: {1: 0.5176817288801572, 0: 0.4823182711198428}


## Compute MAX_LEN from 95th percentile

We compute sequence lengths (premise + hypothesis) and set `MAX_LEN` to the **95th percentile** (rounded to a multiple of 5).
This retains most training examples while limiting unnecessary padding and reducing computational cost.

In [13]:
# Max length was chosen from this value
prem_lens = train_df["premise"].astype(str).str.split().str.len().values
hyp_lens  = train_df["hypothesis"].astype(str).str.split().str.len().values

p95 = int(np.percentile(np.concatenate([prem_lens, hyp_lens]), 95))
MAX_LEN = int(np.ceil(p95 / 5) * 5)  # nice rounded length (Maximum length is determined using the 95th percentile of sequence lengths.)

print("Using MAX_LEN:", MAX_LEN)

Using MAX_LEN: 35


## Build Tokeniser

The tokenizer builds a vocabulary from the training premise and hypothesis texts and maps each word to an integer index so the model can process the input sequences.

In [14]:
# Create and fit tokenizer on combined premise and hypothesis text from the training set
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")

# Fit tokenizer on both premise and hypothesis text
tokenizer.fit_on_texts(
    train_premises.tolist() + train_hypotheses.tolist()
)

# Show tokenizer statistics
print("Tokenizer vocabulary size:", len(tokenizer.word_index))
print("Configured VOCAB_SIZE:", VOCAB_SIZE)

Tokenizer vocabulary size: 35033
Configured VOCAB_SIZE: 30000


The text is converted into sequences of word IDs and padded to a fixed length so it can be used as input to the model.

In [15]:
def tokenize_and_pad(texts, tokenizer, max_length):
    """
    Convert text to sequences and pad to fixed length.
    """
    sequences = tokenizer.texts_to_sequences(texts.astype(str).tolist())
    return pad_sequences(
        sequences,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )


# Training data
X_train_p = tokenize_and_pad(train_premises, tokenizer, MAX_LEN)
X_train_h = tokenize_and_pad(train_hypotheses, tokenizer, MAX_LEN)

# Development data
X_dev_p = tokenize_and_pad(dev_premises, tokenizer, MAX_LEN)
X_dev_h = tokenize_and_pad(dev_hypotheses, tokenizer, MAX_LEN)

print("X_train_p:", X_train_p.shape, "X_train_h:", X_train_h.shape, "y_train:", y_train.shape)
print("X_dev_p  :", X_dev_p.shape,   "X_dev_h  :", X_dev_h.shape,   "y_dev  :", y_dev.shape)

X_train_p: (24432, 35) X_train_h: (24432, 35) y_train: (24432,)
X_dev_p  : (6736, 35) X_dev_h  : (6736, 35) y_dev  : (6736,)


## Save Tokenizer

The tokenizer configuration is saved so the same preprocessing and settings can be reused during evaluation and prediction.

In [16]:
# Save the tokenizer so the same vocabulary can be used during evaluation
tok_path = os.path.join(OUT_DIR, "tokenizer.pkl")
with open(tok_path, "wb") as f:
    pickle.dump(tokenizer, f)



## Load FastText vectors

The FastText Wiki News 300-dimensional embeddings are used as pretrained word vectors to capture semantic meaning and handle rare or unseen words using subword information.

In [17]:
# Load pretrained FastText word vectors (300-dimensional embeddings)
ft = api.load("fasttext-wiki-news-subwords-300")

print("✅ FastText loaded!")

[==================================================] 100.0% 958.5/958.4MB downloaded
✅ FastText loaded!


## Build embedding matrix

A function is used to construct the embedding matrix by mapping each word in the tokenizer vocabulary to its corresponding FastText vector. Words not present in the pretrained embedding model remain initialised as zero vectors.

In [18]:
def build_embedding_matrix(tokenizer, embedding_model, vocab_size, emb_dim):
    """
    Create embedding matrix mapping tokenizer words to pretrained vectors.
    Words not found in the embedding model remain zero vectors.
    """

    word_index = tokenizer.word_index
    num_words = min(vocab_size, len(word_index) + 1)

    embedding_matrix = np.zeros((num_words, emb_dim), dtype=np.float32)

    hits = 0

    for word, idx in word_index.items():

        if idx >= num_words:
            continue

        if word in embedding_model:
            embedding_matrix[idx] = embedding_model[word]
            hits += 1

    print(f"Embedding hits: {hits}/{num_words} ({hits/num_words:.2%})")

    return embedding_matrix, num_words

In [19]:
embedding_matrix, num_words = build_embedding_matrix(
    tokenizer,
    ft,
    VOCAB_SIZE,
    EMBED_DIM
)

Embedding hits: 24856/30000 (82.85%)


## Build NLI Model with BiLSTM and Soft Attention

The model follows an ESIM-style architecture using pretrained FastText embeddings and a shared BiLSTM encoder to generate contextual representations for the premise and hypothesis.

A soft attention alignment mechanism is then applied to identify important interactions between the two sentences.

The model constructs local inference features using concatenation, difference, and element-wise multiplication of the encoded and aligned representations.

Finally, the token-level features are summarised using average and max pooling and passed through a multi-layer perceptron classifier to predict the binary entailment label.

In [20]:
def build_model(max_len: int, num_words: int, emb_dim: int, embedding_matrix: np.ndarray):
    # Two inputs: token IDs for premise and hypothesis
    prem_in = Input(shape=(max_len,), name="premise")
    hyp_in  = Input(shape=(max_len,), name="hypothesis")

    # Pretrained FastText embedding layer
    emb = Embedding(
        input_dim=num_words,
        output_dim=emb_dim,
        weights=[embedding_matrix],
        trainable=False,
        mask_zero=False,
        name="fasttext_emb"
    )

    prem = emb(prem_in)
    hyp  = emb(hyp_in)

    # Single shared BiLSTM encoder
    enc = Bidirectional(LSTM(200, return_sequences=True, dropout=0.20), name="enc")
    A = enc(prem)
    B = enc(hyp)

    # Normalise contextual representations
    A = LayerNormalization()(A)
    B = LayerNormalization()(B)

    # Soft alignment between premise and hypothesis
    align = Attention(name="soft_align")
    A_tilde = align([A, B])
    B_tilde = align([B, A])

    # Local inference features
    A_feat = Concatenate()([A, A_tilde, Subtract()([A, A_tilde]), Multiply()([A, A_tilde])])
    B_feat = Concatenate()([B, B_tilde, Subtract()([B, B_tilde]), Multiply()([B, B_tilde])])

    # Compare aligned features
    comp = Dense(200, activation="relu")
    A_comp = Dropout(0.25)(comp(A_feat))
    B_comp = Dropout(0.25)(comp(B_feat))

    # Pool token-level features into sentence-level vectors
    A_avg = GlobalAveragePooling1D()(A_comp)
    A_max = GlobalMaxPooling1D()(A_comp)
    B_avg = GlobalAveragePooling1D()(B_comp)
    B_max = GlobalMaxPooling1D()(B_comp)

    x = Concatenate()([A_avg, A_max, B_avg, B_max])

    # MLP classifier head
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.40)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.30)(x)

    out = Dense(1, activation="sigmoid")(x)
    return Model([prem_in, hyp_in], out)

In [21]:
# Build the model
model = build_model(MAX_LEN, num_words, EMBED_DIM, embedding_matrix)

# Print model architecture
model.summary()
print("Total parameters:", model.count_params())

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ premise             │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hypothesis          │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fasttext_emb        │ (None, 35, 300)   │  9,000,000 │ premise[0][0],    │
│ (Embedding)         │                   │            │ hypothesis[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc (Bidirectional) │ (None, 35, 400)   │    801,600 │ fasttext_emb[0][… │
│                     │                   │            │ fasttext_emb[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 35, 400)   │        800 │ enc[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 35, 400)   │        800 │ enc[1][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ soft_align          │ (None, 35, 400)   │          0 │ layer_normalizat… │
│ (Attention)         │                   │            │ layer_normalizat… │
│                     │                   │            │ layer_normalizat… │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract (Subtract) │ (None, 35, 400)   │          0 │ layer_normalizat… │
│                     │                   │            │ soft_align[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 35, 400)   │          0 │ layer_normalizat… │
│                     │                   │            │ soft_align[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ subtract_1          │ (None, 35, 400)   │          0 │ layer_normalizat… │
│ (Subtract)          │                   │            │ soft_align[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 35, 400)   │          0 │ layer_normalizat… │
│ (Multiply)          │                   │            │ soft_align[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 35, 1600)  │          0 │ layer_normalizat… │
│ (Concatenate)       │                   │            │ soft_align[0][0], │
│                     │                   │            │ subtract[0][0],   │
│                     │                   │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 35, 1600)  │          0 │ layer_normalizat… │
│ (Concatenate)       │                   │            │ soft_align[1][0], │
│                     │                   │            │ subtract_1[0][0], │
│                     │                   │            │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 35, 200)   │    320,200 │ concatenate[0][0… │
│                     │                   │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 35, 200)   │          0 │ dense[0][0]     

 Total params: 10,361,481 (39.53 MB)

 Trainable params: 1,361,481 (5.19 MB)

 Non-trainable params: 9,000,000 (34.33 MB)

Total parameters: 10361481


## Compile Model

The model is compiled using the Adam optimizer with gradient clipping, and trained using callbacks for early stopping, learning rate reduction, and saving the best model.

In [22]:
# Compile the model using Adam optimizer with gradient clipping
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=8e-4, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Training callbacks to improve training stability and model selection
callbacks = [

    # Stop training early if validation loss does not improve
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),

    # Reduce learning rate when validation loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
        verbose=1
    ),

    # Save the best performing model during training
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUT_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True
    ),
]


## Train Model

In [23]:
# Train the model on the training data and validate on the dev set
history = model.fit(
    x=[X_train_p, X_train_h],
    y=y_train,
    validation_data=([X_dev_p, X_dev_h], y_dev),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/15
764/764 ━━━━━━━━━━━━━━━━━━━━ 30s 25ms/step - accuracy: 0.5967 - loss: 0.6669 - val_accuracy: 0.6461 - val_loss: 0.6352 - learning_rate: 8.0000e-04
Epoch 2/15
764/764 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.6518 - loss: 0.6181 - val_accuracy: 0.6676 - val_loss: 0.6085 - learning_rate: 8.0000e-04
Epoch 3/15
764/764 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - accuracy: 0.6581 - loss: 0.6086 - val_accuracy: 0.6731 - val_loss: 0.5978 - learning_rate: 8.0000e-04
Epoch 4/15
764/764 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.6730 - loss: 0.5961 - val_accuracy: 0.6915 - val_loss: 0.5790 - learning_rate: 8.0000e-04
Epoch 5/15
763/764 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6838 - loss: 0.5844
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00039999998989515007.
764/764 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.6852 - loss: 0.5817 - val_accuracy: 0.6918 - val_loss: 0.5801 - learning_rate: 8.0000e-04
Epoch 6/15
764/764 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - a

## Save Training Configuration

The training hyperparameters and model settings are saved to a JSON file so the experiment can be reproduced during evaluation.

In [24]:
# Save model configuration for reproducibility
cfg = {
    "VOCAB_SIZE": VOCAB_SIZE,
    "tokenizer_vocab_size": len(tokenizer.word_index),
    "EMBED_DIM": EMBED_DIM,
    "MAX_LEN": MAX_LEN,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "model": "ESIM-lite (BiLSTM + Attention align + compare + pool)",
    "embedding": "fasttext-wiki-news-subwords-300",
    "embedding_trainable": False,
    "optimizer": "Adam",
    "learning_rate": 8e-4,
    "clipnorm": 1.0
}

cfg_path = os.path.join(OUT_DIR, "config.json")
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

print("✅ Saved config:", cfg_path)
print("✅ Best checkpoint:", os.path.join(OUT_DIR, "best_model.keras"))

✅ Saved config: ./outputs/category_B/config.json
✅ Best checkpoint: ./outputs/category_B/best_model.keras
